Convert .mat to .hdf5

Written by GPT-5.6 Sol

In [12]:
from pathlib import Path
from datetime import datetime, timezone

import h5py
import numpy as np
from scipy.io import loadmat


# =========================================================================
# Configuration
# =========================================================================

ROOT_DIR = Path(
    r"C:\Users\JLU-SU\GitHub\pep_wp4_eye_tracking\sourcedata"
)

OVERWRITE = False

# gzip 4 is usually a good speed/size compromise
GZIP_LEVEL = 4


# =========================================================================
# Basic HDF5 helpers
# =========================================================================

def numeric_dataset(group, name, value, transpose_samples=False):
    """
    Write a numerical MATLAB array.

    transpose_samples=True converts MATLAB-style D x N arrays into
    sample-major N x D arrays.
    """

    arr = np.asarray(value)

    # Remove singleton dimensions such as 1 x N
    arr = np.squeeze(arr)

    if (
        transpose_samples
        and arr.ndim == 2
        and arr.shape[0] in (2, 3)
        and arr.shape[1] > arr.shape[0]
    ):
        original_shape = arr.shape
        arr = arr.T
    else:
        original_shape = arr.shape

    # MATLAB logical values may arrive as uint8 or bool.
    # uint8 is highly portable and compresses well.
    if arr.dtype == np.bool_:
        arr = arr.astype(np.uint8)

    kwargs = {}

    if arr.size >= 1000:
        kwargs = {
            "compression": "gzip",
            "compression_opts": GZIP_LEVEL,
            "shuffle": True,
            "chunks": True,
        }

    ds = group.create_dataset(
        name,
        data=arr,
        **kwargs,
    )

    ds.attrs["original_matlab_shape"] = original_shape

    return ds


def string_dataset(group, name, values):

    dtype = h5py.string_dtype("utf-8")

    values = np.asarray(values, dtype=object)

    return group.create_dataset(
        name,
        data=values,
        dtype=dtype,
    )


# =========================================================================
# Gaze conversion
# =========================================================================

def write_eye(group, eye):
    """
    Convert one eye:
        gazePoint
        pupil
        gazeOrigin
        eyeOpenness
    """

    # ---------------------------------------------------------------------
    # Gaze point
    # ---------------------------------------------------------------------

    gp = group.create_group("gaze_point")

    numeric_dataset(
        gp,
        "on_display_area",
        eye["gazePoint"]["onDisplayArea"],
        transpose_samples=True,
    )

    numeric_dataset(
        gp,
        "in_user_coords",
        eye["gazePoint"]["inUserCoords"],
        transpose_samples=True,
    )

    numeric_dataset(
        gp,
        "valid",
        eye["gazePoint"]["valid"],
    )

    numeric_dataset(
        gp,
        "available",
        eye["gazePoint"]["available"],
    )


    # ---------------------------------------------------------------------
    # Pupil
    # ---------------------------------------------------------------------

    pupil = group.create_group("pupil")

    numeric_dataset(
        pupil,
        "diameter",
        eye["pupil"]["diameter"],
    )

    numeric_dataset(
        pupil,
        "valid",
        eye["pupil"]["valid"],
    )

    numeric_dataset(
        pupil,
        "available",
        eye["pupil"]["available"],
    )


    # ---------------------------------------------------------------------
    # Gaze origin
    # ---------------------------------------------------------------------

    origin = group.create_group("gaze_origin")

    numeric_dataset(
        origin,
        "in_user_coords",
        eye["gazeOrigin"]["inUserCoords"],
        transpose_samples=True,
    )

    numeric_dataset(
        origin,
        "in_track_box_coords",
        eye["gazeOrigin"]["inTrackBoxCoords"],
        transpose_samples=True,
    )

    numeric_dataset(
        origin,
        "valid",
        eye["gazeOrigin"]["valid"],
    )

    numeric_dataset(
        origin,
        "available",
        eye["gazeOrigin"]["available"],
    )


    # ---------------------------------------------------------------------
    # Eye openness
    # ---------------------------------------------------------------------

    openness = group.create_group("eye_openness")

    numeric_dataset(
        openness,
        "diameter",
        eye["eyeOpenness"]["diameter"],
    )

    numeric_dataset(
        openness,
        "valid",
        eye["eyeOpenness"]["valid"],
    )

    numeric_dataset(
        openness,
        "available",
        eye["eyeOpenness"]["available"],
    )


def write_gaze(h5, gaze):

    out = h5.create_group("gaze")

    numeric_dataset(
        out,
        "device_time_stamp",
        gaze["deviceTimeStamp"],
    )

    numeric_dataset(
        out,
        "system_time_stamp",
        gaze["systemTimeStamp"],
    )

    left = out.create_group("left")
    right = out.create_group("right")

    write_eye(left, gaze["left"])
    write_eye(right, gaze["right"])

    # Determine number of samples
    n_samples = np.asarray(
        gaze["systemTimeStamp"]
    ).size

    out.attrs["n_samples"] = n_samples


# =========================================================================
# Messages
# =========================================================================

def write_messages(h5, messages):

    out = h5.create_group("messages")

    messages = np.asarray(messages, dtype=object)

    # First column = timestamp
    timestamps = np.array(
        [
            np.asarray(x).squeeze()
            for x in messages[:, 0]
        ],
        dtype=np.int64,
    )

    # Second column = text
    text = np.array(
        [
            str(x)
            for x in messages[:, 1]
        ],
        dtype=object,
    )

    numeric_dataset(
        out,
        "timestamp",
        timestamps,
    )

    string_dataset(
        out,
        "text",
        text,
    )


# =========================================================================
# Generic storage for small metadata structures
# =========================================================================

def write_metadata_value(group, name, value):
    """
    Generic writer intended for SMALL metadata structures only.

    This is deliberately not used for the main gaze arrays.
    """

    if value is None:
        g = group.create_group(name)
        g.attrs["empty"] = True
        return

    if isinstance(value, dict):

        g = group.create_group(name)

        for key, val in value.items():
            write_metadata_value(
                g,
                str(key),
                val,
            )

        return

    if isinstance(value, (str, np.str_)):
        string_dataset(group, name, str(value))
        return

    if isinstance(
        value,
        (bool, int, float, complex, np.number),
    ):
        group.create_dataset(name, data=value)
        return

    if isinstance(value, np.ndarray):

        if value.dtype.kind in "biufc":
            numeric_dataset(group, name, value)
            return

        if value.dtype.kind in "SU":
            string_dataset(
                group,
                name,
                value.astype(str),
            )
            return

        # Small MATLAB cell arrays
        if value.dtype == object:

            g = group.create_group(name)
            g.attrs["matlab_shape"] = value.shape

            for i, item in enumerate(value.flat):
                write_metadata_value(
                    g,
                    f"item_{i + 1:04d}",
                    item,
                )

            return

    if isinstance(value, (list, tuple)):

        # Homogeneous strings
        if all(
            isinstance(v, (str, np.str_))
            for v in value
        ):
            string_dataset(group, name, value)
            return

        # Homogeneous numeric scalars
        if all(
            isinstance(v, (bool, int, float, np.number))
            for v in value
        ):
            numeric_dataset(group, name, value)
            return

        # Struct array: small metadata such as TobiiLog
        if all(isinstance(v, dict) for v in value):

            g = group.create_group(name)

            g.attrs["representation"] = "struct_array"
            g.attrs["length"] = len(value)

            # Use the UNION of all fields, because not every struct element
            # necessarily contains exactly the same fields.
            fields = sorted(
                set().union(
                    *(row.keys() for row in value)
                )
            )

            for field in fields:

                # Some struct elements may not contain this field.
                vals = [
                    row.get(field, None)
                    for row in value
                ]

                present = np.array(
                    [
                        field in row and row[field] is not None
                        for row in value
                    ],
                    dtype=np.uint8,
                )

                nonmissing = [
                    x
                    for x in vals
                    if x is not None
                ]

                # -------------------------------------------------------------
                # Field absent everywhere
                # -------------------------------------------------------------

                if len(nonmissing) == 0:

                    field_group = g.create_group(str(field))
                    field_group.attrs["empty"] = True
                    continue

                # -------------------------------------------------------------
                # Strings
                # -------------------------------------------------------------

                if all(
                    isinstance(x, (str, np.str_))
                    for x in nonmissing
                ):

                    # Empty string marks missing entries.
                    # A separate "_present" mask records which entries
                    # actually contained the field.
                    packed = [
                        "" if x is None else str(x)
                        for x in vals
                    ]

                    string_dataset(
                        g,
                        str(field),
                        packed,
                    )

                    if not np.all(present):
                        numeric_dataset(
                            g,
                            f"{field}_present",
                            present,
                        )

                    continue

                # -------------------------------------------------------------
                # Numeric scalars
                # -------------------------------------------------------------

                if all(
                    np.asarray(x).size == 1
                    and np.asarray(x).dtype.kind in "biuf"
                    for x in nonmissing
                ):

                    # Use NaN for missing numerical values.
                    # Float is necessary because integer arrays cannot store NaN.
                    if np.all(present):

                        packed = np.asarray([
                            np.asarray(x).item()
                            for x in vals
                        ])

                    else:

                        packed = np.full(
                            len(vals),
                            np.nan,
                            dtype=float,
                        )

                        for i, x in enumerate(vals):

                            if x is not None:
                                packed[i] = np.asarray(x).item()

                    numeric_dataset(
                        g,
                        str(field),
                        packed,
                    )

                    if not np.all(present):
                        numeric_dataset(
                            g,
                            f"{field}_present",
                            present,
                        )

                    continue

                # -------------------------------------------------------------
                # More complicated / heterogeneous values
                # -------------------------------------------------------------

                field_group = g.create_group(str(field))

                field_group.attrs["representation"] = (
                    "heterogeneous_struct_field"
                )

                for i, item in enumerate(vals):

                    write_metadata_value(
                        field_group,
                        f"item_{i + 1:04d}",
                        item,
                    )

            return

        g = group.create_group(name)

        for i, item in enumerate(value):
            write_metadata_value(
                g,
                f"item_{i + 1:04d}",
                item,
            )

        return

    raise TypeError(
        f"Unsupported metadata value {name}: {type(value)}"
    )


# =========================================================================
# Convert one participant
# =========================================================================

def convert_file(mat_file, h5_file):

    print()
    print(f"[START] {mat_file.name}")

    mat = loadmat(
        mat_file,
        simplify_cells=True,
    )

    mat = {
        key: value
        for key, value in mat.items()
        if not key.startswith("__")
    }

    if h5_file.exists():

        if OVERWRITE:
            h5_file.unlink()

        else:
            print("        HDF5 already exists -- skipping.")
            return


    with h5py.File(h5_file, "w") as h5:

        # -----------------------------------------------------------------
        # File metadata
        # -----------------------------------------------------------------

        h5.attrs["source_mat_file"] = mat_file.name

        h5.attrs["conversion_date_utc"] = (
            datetime.now(timezone.utc).isoformat()
        )

        h5.attrs["format_description"] = (
            "Eye-tracking data converted from MATLAB "
            "to open HDF5 representation"
        )


        # -----------------------------------------------------------------
        # Main gaze data
        # -----------------------------------------------------------------

        write_gaze(
            h5,
            mat["data"]["gaze"],
        )


        # -----------------------------------------------------------------
        # Messages
        # -----------------------------------------------------------------

        write_messages(
            h5,
            mat["messages"],
        )


        # -----------------------------------------------------------------
        # Remaining acquisition data
        # -----------------------------------------------------------------

        acquisition = h5.create_group(
            "acquisition"
        )

        for name in [
            "eyeImages",
            "externalSignals",
            "timeSync",
            "notifications",
        ]:

            if name in mat["data"]:
                write_metadata_value(
                    acquisition,
                    name,
                    mat["data"][name],
                )


        # -----------------------------------------------------------------
        # Other metadata
        # -----------------------------------------------------------------

        for name in [
            "calibration",
            "systemInfo",
            "geometry",
            "settings",
            "TobiiLog",
            "expt",
        ]:

            if name in mat:
                write_metadata_value(
                    h5,
                    name,
                    mat[name],
                )


    mat_mb = mat_file.stat().st_size / 1024**2
    h5_mb = h5_file.stat().st_size / 1024**2

    print(f"        MAT:  {mat_mb:.1f} MB")
    print(f"        HDF5: {h5_mb:.1f} MB")
    print(f"[DONE]  {h5_file.name}")


# =========================================================================
# Loop over subjects
# =========================================================================

subject_dirs = sorted(
    p
    for p in ROOT_DIR.glob("sub-*")
    if p.is_dir()
)

print(
    f"Found {len(subject_dirs)} subject directories."
)


for subject_dir in subject_dirs:

    subject = subject_dir.name

    mat_file = (
        subject_dir
        / f"{subject}_task-EyeTracking_physio.mat"
    )

    h5_file = (
        subject_dir
        / f"{subject}_task-EyeTracking_physio.hdf5"
    )

    if not mat_file.exists():
        print(f"[SKIP] {subject}: MAT file missing.")
        continue

    convert_file(
        mat_file,
        h5_file,
    )

print()
print("Conversion finished.")

Found 72 subject directories.

[START] sub-001_task-EyeTracking_physio.mat
        HDF5 already exists -- skipping.

[START] sub-002_task-EyeTracking_physio.mat
        HDF5 already exists -- skipping.

[START] sub-003_task-EyeTracking_physio.mat
        MAT:  21.0 MB
        HDF5: 23.5 MB
[DONE]  sub-003_task-EyeTracking_physio.hdf5

[START] sub-004_task-EyeTracking_physio.mat
        MAT:  16.2 MB
        HDF5: 18.0 MB
[DONE]  sub-004_task-EyeTracking_physio.hdf5

[START] sub-005_task-EyeTracking_physio.mat
        MAT:  17.1 MB
        HDF5: 19.4 MB
[DONE]  sub-005_task-EyeTracking_physio.hdf5

[START] sub-006_task-EyeTracking_physio.mat
        MAT:  16.8 MB
        HDF5: 18.9 MB
[DONE]  sub-006_task-EyeTracking_physio.hdf5

[START] sub-007_task-EyeTracking_physio.mat
        MAT:  16.2 MB
        HDF5: 18.8 MB
[DONE]  sub-007_task-EyeTracking_physio.hdf5

[START] sub-008_task-EyeTracking_physio.mat
        MAT:  16.5 MB
        HDF5: 17.4 MB
[DONE]  sub-008_task-EyeTracking_physio

Check new .hdf5 files

In [ ]:
from pathlib import Path

import h5py
import numpy as np
from scipy.io import loadmat


# -------------------------------------------------------------------------
# Configuration
# -------------------------------------------------------------------------

ROOT_DIR = Path(
    r"C:\Users\JLU-SU\GitHub\pep_wp4_eye_tracking\sourcedata"
)

# Set to e.g. ["sub-001"] to test only one subject.
# Set to None to test all subjects.
SUBJECTS = None


# -------------------------------------------------------------------------
# Helper
# -------------------------------------------------------------------------

def compare_array(mat_value, h5_value, name, transpose=False):

    a = np.asarray(mat_value).squeeze()
    b = np.asarray(h5_value)

    # Coordinate matrices were converted from D x N to N x D
    if transpose and a.ndim == 2:
        a = a.T

    shape_ok = a.shape == b.shape

    if not shape_ok:
        print(
            f"  [FAIL] {name}: "
            f"shape MAT={a.shape}, HDF5={b.shape}"
        )
        return False

    # exact comparison for integer/logical data
    if a.dtype.kind in "biu":
        values_ok = np.array_equal(a, b)

    # tolerance-based comparison for floating point
    else:
        values_ok = np.allclose(
            a,
            b,
            rtol=1e-10,
            atol=1e-12,
            equal_nan=True,
        )

    if values_ok:
        print(f"  [PASS] {name} {a.shape}")
        return True

    else:
        # useful diagnostic
        valid = np.isfinite(a) & np.isfinite(b)

        if np.any(valid):
            max_diff = np.max(
                np.abs(a[valid] - b[valid])
            )
        else:
            max_diff = np.nan

        print(
            f"  [FAIL] {name}: "
            f"values differ; max diff = {max_diff}"
        )
        return False


# -------------------------------------------------------------------------
# Check one subject
# -------------------------------------------------------------------------

def check_subject(subject_dir):

    subject = subject_dir.name

    mat_file = (
        subject_dir
        / f"{subject}_task-EyeTracking_physio.mat"
    )

    h5_file = (
        subject_dir
        / f"{subject}_task-EyeTracking_physio.hdf5"
    )

    print()
    print("=" * 70)
    print(subject)
    print("=" * 70)

    if not mat_file.exists():
        print("[FAIL] MAT file missing")
        return False

    if not h5_file.exists():
        print("[FAIL] HDF5 file missing")
        return False

    mat = loadmat(
        mat_file,
        simplify_cells=True,
    )

    gaze = mat["data"]["gaze"]

    results = []

    with h5py.File(h5_file, "r") as h5:

        # -------------------------------------------------------------
        # Timestamps
        # -------------------------------------------------------------

        results.append(
            compare_array(
                gaze["deviceTimeStamp"],
                h5["gaze/device_time_stamp"][:],
                "device timestamp",
            )
        )

        results.append(
            compare_array(
                gaze["systemTimeStamp"],
                h5["gaze/system_time_stamp"][:],
                "system timestamp",
            )
        )


        # -------------------------------------------------------------
        # Left and right eye
        # -------------------------------------------------------------

        for eye in ["left", "right"]:

            src = gaze[eye]
            dst = h5[f"gaze/{eye}"]

            # gaze point
            results.append(
                compare_array(
                    src["gazePoint"]["onDisplayArea"],
                    dst["gaze_point/on_display_area"][:],
                    f"{eye} gazePoint onDisplayArea",
                    transpose=True,
                )
            )

            results.append(
                compare_array(
                    src["gazePoint"]["inUserCoords"],
                    dst["gaze_point/in_user_coords"][:],
                    f"{eye} gazePoint inUserCoords",
                    transpose=True,
                )
            )

            for field in ["valid", "available"]:

                results.append(
                    compare_array(
                        src["gazePoint"][field],
                        dst[f"gaze_point/{field}"][:],
                        f"{eye} gazePoint {field}",
                    )
                )

            # pupil
            for field in ["diameter", "valid", "available"]:

                results.append(
                    compare_array(
                        src["pupil"][field],
                        dst[f"pupil/{field}"][:],
                        f"{eye} pupil {field}",
                    )
                )

            # gaze origin
            results.append(
                compare_array(
                    src["gazeOrigin"]["inUserCoords"],
                    dst["gaze_origin/in_user_coords"][:],
                    f"{eye} gazeOrigin inUserCoords",
                    transpose=True,
                )
            )

            results.append(
                compare_array(
                    src["gazeOrigin"]["inTrackBoxCoords"],
                    dst[
                        "gaze_origin/in_track_box_coords"
                    ][:],
                    f"{eye} gazeOrigin inTrackBoxCoords",
                    transpose=True,
                )
            )

            for field in ["valid", "available"]:

                results.append(
                    compare_array(
                        src["gazeOrigin"][field],
                        dst[f"gaze_origin/{field}"][:],
                        f"{eye} gazeOrigin {field}",
                    )
                )

            # eye openness
            for field in ["diameter", "valid", "available"]:

                results.append(
                    compare_array(
                        src["eyeOpenness"][field],
                        dst[f"eye_openness/{field}"][:],
                        f"{eye} eyeOpenness {field}",
                    )
                )


        # -------------------------------------------------------------
        # Messages
        # -------------------------------------------------------------

        messages = np.asarray(
            mat["messages"],
            dtype=object,
        )

        mat_times = np.array(
            [
                np.asarray(x).squeeze()
                for x in messages[:, 0]
            ],
            dtype=np.int64,
        )

        h5_times = h5["messages/timestamp"][:]

        results.append(
            compare_array(
                mat_times,
                h5_times,
                "message timestamps",
            )
        )

        mat_text = [
            str(x)
            for x in messages[:, 1]
        ]

        h5_text = [
            x.decode("utf-8")
            if isinstance(x, bytes)
            else str(x)
            for x in h5["messages/text"][:]
        ]

        messages_ok = mat_text == h5_text

        print(
            f"  [{'PASS' if messages_ok else 'FAIL'}] "
            f"message text ({len(mat_text)} messages)"
        )

        results.append(messages_ok)


        # -------------------------------------------------------------
        # Basic sanity checks
        # -------------------------------------------------------------

        n = len(h5["gaze/system_time_stamp"])

        print(f"\n  Gaze samples: {n:,}")

        if "n_samples" in h5["gaze"].attrs:
            attr_n = int(h5["gaze"].attrs["n_samples"])

            ok = attr_n == n
            results.append(ok)

            print(
                f"  [{'PASS' if ok else 'FAIL'}] "
                f"n_samples attribute = {attr_n:,}"
            )


    # -----------------------------------------------------------------
    # Size
    # -----------------------------------------------------------------

    mat_mb = mat_file.stat().st_size / 1024**2
    h5_mb = h5_file.stat().st_size / 1024**2

    print()
    print(f"  MAT size:  {mat_mb:.2f} MB")
    print(f"  HDF5 size: {h5_mb:.2f} MB")
    print(f"  Ratio:     {h5_mb / mat_mb:.2f}x")

    success = all(results)

    print()
    print(
        "[ALL CHECKS PASSED]"
        if success
        else "[ONE OR MORE CHECKS FAILED]"
    )

    return success


# -------------------------------------------------------------------------
# Run
# -------------------------------------------------------------------------

if SUBJECTS is None:

    subject_dirs = sorted(
        p for p in ROOT_DIR.glob("sub-*")
        if p.is_dir()
    )

else:

    subject_dirs = [
        ROOT_DIR / subject
        for subject in SUBJECTS
    ]


all_results = []

for subject_dir in subject_dirs:
    all_results.append(
        check_subject(subject_dir)
    )


print()
print("=" * 70)

print(
    f"Passed: {sum(all_results)} / {len(all_results)} subjects"
)

print("=" * 70)


sub-001
  [PASS] device timestamp (247960,)
  [PASS] system timestamp (247960,)
  [PASS] left gazePoint onDisplayArea (247960, 2)
  [PASS] left gazePoint inUserCoords (247960, 3)
  [PASS] left gazePoint valid (247960,)
  [PASS] left gazePoint available (247960,)
  [PASS] left pupil diameter (247960,)
  [PASS] left pupil valid (247960,)
  [PASS] left pupil available (247960,)
  [PASS] left gazeOrigin inUserCoords (247960, 3)
  [PASS] left gazeOrigin inTrackBoxCoords (247960, 3)
  [PASS] left gazeOrigin valid (247960,)
  [PASS] left gazeOrigin available (247960,)
  [PASS] left eyeOpenness diameter (247960,)
  [PASS] left eyeOpenness valid (247960,)
  [PASS] left eyeOpenness available (247960,)
  [PASS] right gazePoint onDisplayArea (247960, 2)
  [PASS] right gazePoint inUserCoords (247960, 3)
  [PASS] right gazePoint valid (247960,)
  [PASS] right gazePoint available (247960,)
  [PASS] right pupil diameter (247960,)
  [PASS] right pupil valid (247960,)
  [PASS] right pupil available (24